Transformers 라이브러리

Dataset
- KLUE 데이터셋 주제 분류 태스크인 YNAT 데이터셋
- 현재 환경에 datasets 설치: pip install datasets==2.20.0
- 학습 데이터 45,678건 / 검증 데이터 9,107건
- 'guid', 'title', 'label', 'url', 'date' 5개 컬럼으로 구성
- label 분류를 위한 정답 값으로 IT 과학 0/ 경제 1/ 사회 2/ 생활문화 3/ 세계 4/ 스포츠 5/ 정치 6 구성됨

Transformers 설치
- NumPy 1.26.x, PyTorch 2.2.2, Transformers 4.44.2 버전

Tokenizer 기본
- BertTokenize: KLUE 데이터셋으로 학습한 klue/bert-base 모델 사용
- tokenizer 인코딩: 문자열 토큰화 작업 -> 토큰을 입력 식별자로 변환
- tokenizer 디코딩: 토큰화 완료 -> 사용자 문자열 변환

Dataset 전처리
- Dataset에 활용한 KLUE 데이터셋을 토크나이저로 전처리(KLUE 데이터셋 YNAT 데이터 로드)
- 전체 데이터셋 토큰화 과정
- 전체 데이터셋을 배치로 나누어 토큰화 과정

DataCollator
- Hugging Face 트랜스포머 라이브러리에서 제공하는 핵심 유틸리티
- 여러개의 데이터 샘플(딕셔너리)을 모아서 모델에 투입할 수 있는 하나의 미니배치(mini-batch)로 조립해 주는 역할
- 전처리는 짦은 상태 그대로 가볍게 끝내고, 미니배치로 묶는 바로 그 순간에 그 배치 안에서 가장 긴 문장의 길이를 확인
- 그리고 그 배치에 속한 문장들만 딱 그 길이 만큼 빈 공간을 [PAD]로 채워준다(메모리 낭비가 없고 학습 속도가 훨씬 빨라짐)

Model 다운로드
- PretrainedModel.from_pretrained 메서드 캐시에 다운로드
- PretrainedModel.save_pretrained 메서드 파일로 저장
- Huggingface_hub 라이브러리 활용하여 프로그래밍으로 다운로드

Model 추론
- BertForMaskedLM 마스킹된 단어를 추론
- AutoClass 사용: BertForMaskedLM 마스킹된 단어를 추론
- Pipeline 사용: 토크나이저 생성, 텐서 변환, 모델 추론, 디코딩 등의 복잡한 과정을 알아서 내부적으로 싹 감싸서(추상화하여) 처리해 주는 편리한 도구

Model 학습
- 토크나이저와 모델 준비
- 데이터 준비 및 전처리
- 학습 파라미터 선정
- 학습 진행
- 성능 평가
- 모델 저장
- 모델 로드

허깅페이스 허브 등록
- 베이스 모델을 파인튜닝(미세조정)하여 로컬에 저장한 나만의 모델을 허깅페이스 허브에 등록하여 모델을 공유 및 재사용(다운로드)
- push_to_hub()
- 허깅페이스 리포지터리 생성
- 허깅페이스 리포지터리 삭제
- 허깅페이스 모델 업로드
- 허깅페이스 모델 다운로드 테스트

Tokenizers 라이브러리
- Tokenizer 학습

Evaluate 라이브러리
- Evaluate 평가
- 커스텀 메트릭
- Trainer 적용

In [1]:
# 시스템 경로
import sys
print(sys.executable)
print(sys.version)

# transformer 설치 확인
import torch
import transformers

print(f"Transformers 버전: {transformers.__version__}")
print(f"PyTorch 버전: {torch.__version__}")
print(f"사용 디바이스: {'cuda' if torch.cuda.is_available() else 'cpu'}")

/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/bin/python
3.11.9 (v3.11.9:de54cf5be3, Apr  2 2024, 07:12:50) [Clang 13.0.0 (clang-1300.0.29.30)]
Transformers 버전: 4.44.2
PyTorch 버전: 2.2.2
사용 디바이스: cpu


In [2]:
# KLUE 데이터셋 주제 분류 태스크인 YNAT 데이터셋 load
from datasets import load_dataset

# klue/klue 경로 및 ynat 서브셋 지정
dataset = load_dataset("klue/klue", "ynat", trust_remote_code=True)

# 결과 확인
print(dataset)
print("\n첫 번째 샘플:")
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9107
    })
})

첫 번째 샘플:
{'guid': 'ynat-v1_train_00000', 'title': '유튜브 내달 2일까지 크리에이터 지원 공간 운영', 'label': 3, 'url': 'https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=105&sid2=227&oid=001&aid=0008508947', 'date': '2016.06.30. 오전 10:36'}


In [3]:
# 비율로 데이터셋 불러오기
from datasets import load_dataset

ratio = 80
dataset = load_dataset(
    "klue/klue",
    "ynat",
    # path="csv",
    # data_files="sample.csv",
    split={
        "train": f"train[:{ratio}%]",
        "test": f"train[{ratio}%:]",
    },
    trust_remote_code=True
)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 36542
    })
    test: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9136
    })
})


In [4]:
# KLUE 데이터셋으로 학습한 klue/bert-base 모델 사용 
from transformers import BertTokenizer

# 토크나이저를 다운로드하여 tokenizer 변수에 BertTokenizer 객체를 할당
model_name = "klue/bert-base"
tokenizer = BertTokenizer.from_pretrained(model_name)

# help 구문으로 tokenizer 변수에 담긴 Tokenizer 객체의 설명
# - BertTokenizer는 PretrainedTokenizer를 상속받고 있으며 워드피스에 기반하고 있다
# help(tokenizer) 

# tokenizer.vocab_size 학습한 사전 크기, tokenizer.get_vocab 토큰 사전 확인, tokenizer.special_tokens_map 새로운 특수 토큰
print(tokenizer.vocab_size)
print(tokenizer.get_vocab)
print(tokenizer.special_tokens_map)

32000
<bound method BertTokenizer.get_vocab of BertTokenizer(name_or_path='klue/bert-base', vocab_size=32000, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}>
{'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mas

/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:
# 토큰화 프로세스: 인코딩
# Tokenizer: KLUE 데이터셋으로 학습한 klue/bert-base 모델 사용
from transformers import BertTokenizer
import warnings

# Transformers 의 경고(FutureWarning) 메시지 숨기기
warnings.filterwarnings("ignore", category=FutureWarning)

# pretrained 모델 다운로드
model_name = "klue/bert-base"
tokenizer = BertTokenizer.from_pretrained(model_name)

sentence = "안녕하세요. 이거 테스트 문장입니다."

# tokenize 메서드: 문자열 토큰리스트 반환
token1 = tokenizer.tokenize(sentence) 
print(f"tokenize 메서드: {token1}")

# convert_tokens_to_ids 메서드: 토큰을 정수 리스트인 토큰 아이디로 반환
ids1 = tokenizer.convert_tokens_to_ids(token1)
print(f"convert_tokens_to_ids 메서드: {ids1}")

# tokenizer 객체: input_ids, token_type_ids, attention_mask 반환
# input_ids: 문장 식별자 ID, 처음과 끝에 각각 2와3 식별자가 붙어 있음을 확인(klue/bert-base 모델에서 시작/종료 [CLS]/[SEP] 특수 번호 포함)
# token_type_ids: 문자열 시쿼스 번호, 입력으로 들어온 텍스트 시퀀스(그룹)의 번호
# attention_mask: 의미가 있으면 1, 없으면 패딩 0, 모델이 연산(Self-Attention)을 할때 진짜 학습/계산해야 할 텍스트 인지(1)/아니면(0) 또는 단순 길이 맞추기용([PAD])
ids2 = tokenizer(sentence)
print(f"tokenizer 객체:{ids2}")

tokenize 메서드: ['안녕', '##하', '##세요', '.', '이거', '테스트', '문장', '##입니다', '.']
convert_tokens_to_ids 메서드: [5891, 2205, 5971, 18, 4647, 7453, 6265, 12190, 18]
tokenizer 객체:{'input_ids': [2, 5891, 2205, 5971, 18, 4647, 7453, 6265, 12190, 18, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [6]:
# 토큰화 프로세스: 디코딩

# ids1([CLS]/[SEP] 특수 토큰이 없는 id(정수) 리스트)
decoded_string1 = tokenizer.decode(ids1)
print(f"decode 메서드1: {decoded_string1}")

# ids2([CLS]/[SEP] 특수 토큰이 포함된 id(정수) 리스트), 
# 토크나이저 객체를 직접 호출해서 얻은 정석적인 input_ids를 디코딩한 것
decoded_string2 = tokenizer.decode(ids2['input_ids'])
print(f"decode 메서드2: {decoded_string2}")

# ids3([CLS]/[SEP] 특수 토큰이 포함된 id(정수) 리스트)
# 토크나이저 객체를 직접 호출해서 얻은 정석적인 input_ids를 디코딩한 것
# input_ids + skip_special_tokens=True (모델 내부 연산용 특수 토큰 [CLS]/[SEP]/[PAD] 등을 제거하고 순수 사용자 텍스트만 추출)
decoded_string3 = tokenizer.decode(ids2['input_ids'], skip_special_tokens=True)
print(f"decode 메서드3: {decoded_string3}")

decode 메서드1: 안녕하세요. 이거 테스트 문장입니다.
decode 메서드2: [CLS] 안녕하세요. 이거 테스트 문장입니다. [SEP]
decode 메서드3: 안녕하세요. 이거 테스트 문장입니다.


In [7]:
# 데이터셋 전처리: 데이터셋 로드(train)
from datasets import load_dataset # 데이터셋 로드 라이브러리

dataset = load_dataset("klue", "ynat") # klue 데이터셋내에 ynat 데이터 로드
raw_train_dataset = dataset["train"] # train 데이터셋

In [8]:
# # 데이터셋 전처리: tokenizer에 데이텃을 입력해 토큰화, 전체 데이터셋 토큰화(실무에서는 전체 데이터를 한번에 사용하지 않음)
# from transformers import BertTokenizer

# # pretrained model 로드(klue/bert-base)
# model_name = "klue/bert-base"
# tokenizer = BertTokenizer.from_pretrained(model_name)

# # KLUE/YNAT 뉴스 데이터셋 제목 리스트 전체를 한꺼번에 토큰화하는 핵심 전처리 과정
# # 모든 문장의 길이를 똑같이 맞추어야 PyTorch 텐서 행렬 연산이 오류 없이 원할하게 돌아가게 하기 위한 과정
# # 단점: 전체 데이터셋을 처리할 충분환 공간의 랜덤 엑세스 메모리가 있는 경우에만 해당된다.
# tokenized_examples = tokenizer(
#     raw_train_dataset["title"], # 입력 데이터
#     padding="max_length", # 배치 내의 모든 문장 길이를 모델이 허용하는 최대 길이로 통일시키는 옵션
#     truncation=True, # 모델이 처리할 수 있는 최대 길이 제한을 초과하는 긴 문장을 허용치 않고 잘라는 옵션
# )

# tokenized_examples

In [9]:
# 데이터셋 전처리: tokenizer에 데이텃을 입력해 토큰화, 전체 데이터셋을 나누어 토큰화(Dataset의 map을 사용하여 배치 단위 토큰화)
from transformers import BertTokenizer

# pretrained model 로드(klue/bert-base)
model_name = "klue/bert-base"
tokenizer = BertTokenizer.from_pretrained(model_name)

# tokenizer 함수
def tokenize_function(sample):
    return tokenizer(sample["title"])

# 전체 데이터셋을 나누어 토큰화
tokenized_dataset = dataset.map(
    tokenize_function, # tokenizer 함수, 데이터세의 일정 묶음(배치) 단위로 뉴스 기사 제목 리스트를 인코딩
    batched=True, # 여러 데이터를 묶어서(batch) 한꺼번에 처리하는 옵션
    batch_size=1000, # 한번에 처리할 데이터 묶음의 단위를 1,000개로 제한하는 옵션
    remove_columns=['guid', 'title', 'url', 'date'], # 토큰화가 끝난후, 더 이상 모델 학습에 필요 없는 원본 컬럼들을 데이터셋에서 삭제
)

# 전체 데이터셋이 전처리 과정을 통해서 업데이트(train/validation) 적용
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9107
    })
})

In [10]:
# DataCollator

# dataset의 map 메서드로 토큰 처리된 배치 내 각 요소의 길이 등 확인
# 파이썬 리스트 형태: 파이토치를 기준으로 학습할때 반드시 torch.Tensor 형태여야 한다
# 파이썬 리스트 형태 그대로 사용하면, torch.Tensor는 다차원 데이터의 모든 길이가 같아야 하기에 오류가 발생할 수 있다
print(tokenized_dataset['train'][0]['input_ids'])
print(type(tokenized_dataset['train'][0]['input_ids']))

[2, 10637, 8474, 22, 2210, 2299, 2118, 28940, 3691, 4101, 3792, 3]
<class 'list'>


In [11]:
# DataCollator: DataCollatorWithPadding(동적패딩 사용) 
# 배치 최대 길이를 기준, torch.Tensor 형태로 타입 변경
from pprint import pprint
from transformers import DataCollatorWithPadding # 동적패딩 

# tokenized_dataset의 train 데이터셋에서 인덱스 0~7번까지의 샘플 8개를 리스트 형태로 추출
batch = [tokenized_dataset["train"][i] for i in range(8)]
# 추출한 8개 샘플 각각의 input_ids 길이(토큰 개수)를 구해서 파이썬 리스트로 만들어 출력
print([len(sample["input_ids"]) for sample in batch])

# DataCollatorWithPadding에 pretrained model(klue/bert-base) 기준
# 1.콜레이터가 8개의 샘플중 가장 긴 토큰 길이를 가진 샘플을 찾아낸다
# 2.나머지 짧은 샘플들의 뒷부분에[PAD] 토큰을 자동으로 채워넣어 모든 샘플의 길이를 가장 긴 샘플에 맞춘다
# 3.파이썬 리스트 형태였던 데이터를 모델 연산에 필요한 PyTorch Tensor(torch.Tensor) 형태로 일괄 변환한다
# 4.attention_mask도 패딩된 위치에 맞춰 알아서 0으로 세팅해준다
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
batch = data_collator(batch)

# 변환된 batch 딕셔너리 안의 각 키(attention_mask,input_ids,labels,token_type_ids)
# 별로 PyTorch 텐서의 크기(Shape)를 출력
pprint({k: v.size() for k, v in batch.items()})

[12, 12, 17, 16, 18, 13, 14, 5]
{'attention_mask': torch.Size([8, 18]),
 'input_ids': torch.Size([8, 18]),
 'labels': torch.Size([8]),
 'token_type_ids': torch.Size([8, 18])}


In [12]:
# PretrainedModel.from_pretrained 메서드 캐시에 다운로드
# BertTokenizer와 BertModel을 활용하여 bert-base-uncased 리포지터리에서 모델 다운로드
from transformers import BertTokenizer, BertModel

# bert-base-uncased 리포지터리 ID 지정: 구글이 영어 대소문자를 구분하지 않고(uncased)학습시킨 기본 Bert 모델
# 처음 실행시 로컬 캐시 디렉토리(~/.cache/huggingface/hub)에 저장
model_name = "bert-base-uncased"

# from_pretrained 메서드로 BertTokenizer 모델 다운로드
tokenizer = BertTokenizer.from_pretrained(model_name)

# from_pretrained 메서드로 BertModel 모델 다운로드
model = BertModel.from_pretrained(model_name)

In [13]:
# PretrainedModel.save_pretrained 메서드 파일로 저장
# 로컬 경로에 토크나이저와 모델 파일로 저장, 보통 미세조정이 완료된 모델을 저장할때 활용
model_path = "./MyBertModel" # 로컬 모델 경로

# save_pretrained 메서드로 tokenizer 저장
tokenizer.save_pretrained(model_path)

# save_pretrained 메서드로 model 저장
model.save_pretrained(model_path)

# 로컬 모델 저장 확인
!ls -l {model_path}

total 855928
-rw-r--r--@ 1 ai  staff        658 Oct 19 21:57 config.json
-rw-------@ 1 ai  staff  437951328 Oct 19 21:57 model.safetensors
-rw-r--r--@ 1 ai  staff        125 Oct 19 21:57 special_tokens_map.json
-rw-r--r--@ 1 ai  staff       1242 Oct 19 21:57 tokenizer_config.json
-rw-r--r--@ 1 ai  staff     231508 Oct 19 21:57 vocab.txt


In [14]:
# Model 추론: BertForMaskedLM 마스킹된 단어를 추론
import torch
from transformers import BertTokenizer, BertForMaskedLM

# 모델명 지정
model_name = "bert-base-uncased"

# BertTokenizer 모델 다운로드
tokenizer = BertTokenizer.from_pretrained(model_name)

# BertForMaskedLM 모델 다운로드
model = BertForMaskedLM.from_pretrained(model_name)

# 1. 입력 문장 토큰화: 자연어 문장을 BERT가 이해할 수 있는 숫자 input_ids로 바꾼다
# 내부적으로 문장속 [MASK]는 BERT의 단어 사정에서 고유한 마스크 토큰 ID(보통 103 등)로 치환된다
text = "The capital of France is [MASK]."
inputs = tokenizer(text, return_tensors="pt") # PyTorch로 텐서(Tensor) 형태로 변환

# 2. 모델 추론: 학습 모드 아님
# with torch.no_grad(): 이미 학습된 가중치로 추론(Inference)만 하는 단계
# 역전파(Backpropagation)용 기울기(Gradient)를 계산하지 않도록 설정하여 메모리를 아끼고 속도를 높인다 
with torch.no_grad():
    # 모델은 단어 사전(Vocabulary)에 있는 약 30,522개의 모든 단어 각각에 대해, 
    # 이 자리에 이 단어가 올 확률이 얼마나 될까? 계산한 점수(Logits)를 출력
    outputs = model(**inputs)
    logits = outputs.logits # Shape: (batch_size, sequence_length, vocab_size)
# logits shape: torch.Size([1, 9, 30522]): 
# 1(batch_size) 입력한 문장이 1개이기 때문에 배치 크기가 1
# 9(sequence_length) "The capital of France is [MASK]." 토크나이저를 거치면서 몇 개의 토큰으로 쪼개졌는지 보여준다
# [CLS] The capital of France is [MASK] . [SEP]
# 30522(vocab_size) 모델이 알고 있는 영어 단어 사전의 총개수
# print(f"outputs: {outputs}")
# print(f"logits: {logits}")
print(f"logits shape: {logits.shape}")

# 3. [MASK] 토큰의 정확한 위치(인덱스) 찾기
# input_ids 중에서 [MASK] 아이디와 일치하는 위치의 인덱스를 찾는다
# torch.where(): 파이토치에서 조건이 True인 곳의 좌표(행과 열 번호)를 찾아내주는 함수
# [1]: torch.where가 반환한 튜플 (행 텐서, 열 텐서) 중 두 번째 값(인덱스 1)인 열(Column) 번호 텐서만 쏙 뽑아내는 것
mask_token_index = torch.where(inputs.input_ids == tokenizer.mask_token_id)[1]
print(f"input.input_ids: {inputs.input_ids}")
print(f"tokenizer.mask_token_id: {tokenizer.mask_token_id}")
print(f"mask_token_index: {mask_token_index}")

# 4. 해당 [MASK] 위치의 로짓(확률값)에서 가장 확률이 높은 단어 ID 추출
# logits[0, mask_token_index, :]
# 0: 배치(문장 묶음)중 0번째 문장을 선택, 입력한 문장이 1개 배치 이므로 0을 지정
# mask_token_index: 9개의 토큰중 찾아낸[MASK]가 들어 있는 정확한 위치(인덱스 번호)이다, (6번째 토큰)
# : 그 위치에서 단어 사전에 있는 모든 단어(30,522개)의 점수를 전부 다 가져온다
selected_logits = logits[0, mask_token_index, :]
print(f"selected_logits: {selected_logits}")

# selected_logits.argmax(dim=-1): 30,522개의 점수 중에서 가장 높은 점수(가장 확률이 높은 단어)를 골라내느 작업
# argmax(): 텐서 안에서 가장 큰 값이 들어있는 '위치(인덱스 번호)'를 찾아주는 파이토치 함수
# (30,522개의 점수 중 15,230번째 점수가 가장 높다면, 숫자 15230 을 반환)
# dim=-1: 가장 마지막 차원(30,522개의 단어들이 나열된 방향)을 기준으로 가장 큰 값을 찾는다
# 0(첫번째 세로,행), 1(두번째 가로,열), -1(가장 마지막 차원)
# 에시: 2차원 텐서일 때 ([배치 크기, 단어 사전], 예: [1, 30522])
# dim=1: 두 번째 축인 단어 사전 방향을 가리킨다
# dim=-1: 가장 마지막 축인 단어 사전 방향을 가리킨다(이 경우에는 둘의 결과가 완전히 같다)
predicted_token_id = selected_logits.argmax(dim=-1)
print(f"predicted_token_id: {predicted_token_id}")

# 5. tokenizer 디코딩 변환
predicted_token = tokenizer.decode(predicted_token_id)
print(f"추론된 단어: {predicted_token}")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


logits shape: torch.Size([1, 9, 30522])
input.input_ids: tensor([[ 101, 1996, 3007, 1997, 2605, 2003,  103, 1012,  102]])
tokenizer.mask_token_id: 103
mask_token_index: tensor([6])
selected_logits: tensor([[-3.7861, -3.8572, -3.5644,  ..., -2.5593, -3.1093, -4.3820]])
predicted_token_id: tensor([3000])
추론된 단어: paris


In [15]:
# AutoClass 사용: BertForMaskedLM 마스킹된 단어를 추론
# 사전학습된 가중치, 구성, 어휘에 대한 이름, 경로가 지정된 관련 모델을 자동으로 검색하는 구조
# AutoClass: BertTokenizer -> AutoTokenizer, BertForMaskedLM -> AutoModelForMaskedLM
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

# 모델명 지정
model_name = "bert-base-uncased"

# 사전 학습된 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)
# 사전 학습된 모델 로드
model = AutoModelForMaskedLM.from_pretrained(model_name)

# 1. 입력 문장 토큰화: 자연어 문장을 BERT가 이해할 수 있는 숫자 input_ids로 바꾼다
text = "The capital of France is [MASK]."
inputs = tokenizer(text=text, return_tensors="pt")

# 2. 모델 추론: 학습 모드 아님
with torch.no_grad():
    # 모델은 단어 사전(Vocabulary)에 있는 약 30,522개의 모든 단어 각각에 대해, 
    # 이 자리에 이 단어가 올 확률이 얼마나 될까? 계산한 점수(Logits)를 출력
    outputs = model(**inputs)
    logits = outputs.logits # Shape: (batch_size, sequence_length, vocab_size)
print(f"logits shape: {logits.shape}")

# 3. [MASK] 토큰의 정확한 위치(인덱스) 찾기
# input_ids 중에서 [MASK] 아이디와 일치하는 위치의 인덱스를 찾는다
# torch.where(): 파이토치에서 조건이 True인 곳의 좌표(행과 열 번호)를 찾아내주는 함수
# [1]: torch.where가 반환한 튜플 (행 텐서, 열 텐서) 중 두 번째 값(인덱스 1)인 열(Column) 번호 텐서만 쏙 뽑아내는 것
mask_token_index = torch.where(inputs.input_ids == tokenizer.mask_token_id)[1]
print(f"input.input_ids: {inputs.input_ids}")
print(f"tokenizer.mask_token_id: {tokenizer.mask_token_id}")
print(f"mask_token_index: {mask_token_index}")

# 4. 해당 [MASK] 위치의 로짓(확률값)에서 가장 확률이 높은 단어 ID 추출
# logits[0, mask_token_index, :]
# 0: 배치(문장 묶음)중 0번째 문장을 선택, 입력한 문장이 1개 배치 이므로 0을 지정
# mask_token_index: 9개의 토큰중 찾아낸[MASK]가 들어 있는 정확한 위치(인덱스 번호)이다, (6번째 토큰)
# : 그 위치에서 단어 사전에 있는 모든 단어(30,522개)의 점수를 전부 다 가져온다
selected_logits = logits[0, mask_token_index, :]
print(f"selected_logits: {selected_logits}")

# selected_logits.argmax(dim=-1): 30,522개의 점수 중에서 가장 높은 점수(가장 확률이 높은 단어)를 골라내느 작업
# argmax(): 텐서 안에서 가장 큰 값이 들어있는 '위치(인덱스 번호)'를 찾아주는 파이토치 함수
# (30,522개의 점수 중 15,230번째 점수가 가장 높다면, 숫자 15230 을 반환)
# dim=-1: 가장 마지막 차원(30,522개의 단어들이 나열된 방향)을 기준으로 가장 큰 값을 찾는다
# 0(첫번째 세로,행), 1(두번째 가로,열), -1(가장 마지막 차원)
# 에시: 2차원 텐서일 때 ([배치 크기, 단어 사전], 예: [1, 30522])
# dim=1: 두 번째 축인 단어 사전 방향을 가리킨다
# dim=-1: 가장 마지막 축인 단어 사전 방향을 가리킨다(이 경우에는 둘의 결과가 완전히 같다)
predicted_token_id = selected_logits.argmax(dim=-1)
print(f"predicted_token_id: {predicted_token_id}")

# 5. tokenizer 디코딩 변환
predicted_token = tokenizer.decode(predicted_token_id)
print(f"추론된 단어: {predicted_token}")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


logits shape: torch.Size([1, 9, 30522])
input.input_ids: tensor([[ 101, 1996, 3007, 1997, 2605, 2003,  103, 1012,  102]])
tokenizer.mask_token_id: 103
mask_token_index: tensor([6])
selected_logits: tensor([[-3.7861, -3.8572, -3.5644,  ..., -2.5593, -3.1093, -4.3820]])
predicted_token_id: tensor([3000])
추론된 단어: paris


In [16]:
# Pipeline: 토크나이저 생성, 텐서 변환, 모델 추론, 디코딩 등의 복잡한 과정을 알아서 내부적으로 싹 감싸서(추상화하여) 처리해 주는 편리한 도구
# 허깅페이스 허브 모델 가져오기
from transformers import pipeline

# task: 텍스트 분류(text-classification, 예)감성 분석/뉴스 카테고리 분류) 작업을 수행하겠다고 지정
# model: 구글의 기본 영어 BERT 모델(bert-base-uncased)을 로드
pipe = pipeline(task="text-classification", model="google-bert/bert-base-uncased")

# 한국어 뉴스 문장을 파이프라인에 던져넣고, 모델이 이를 어떻게 분류하는지 결과를 출력
print(pipe("유튜브 내달 2일까지 크리에이터 지원 공간 운영"))

# LABEL_0 무작위 점수, 의미 없는 점수(0.65) 발생: 해당 태스크와 언어로 이미 미세조정(Fine-tuning)된 모델로 지정
# [{'label': 'LABEL_0', 'score': 0.6527202725410461}]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[{'label': 'LABEL_1', 'score': 0.5154497027397156}]


In [17]:
# Pipeline: 토크나이저 생성, 텐서 변환, 모델 추론, 디코딩 등의 복잡한 과정을 알아서 내부적으로 싹 감싸서(추상화하여) 처리해 주는 편리한 도구
# 허깅페이스 허브 모델 가져오기
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_name = "./MyBertModel"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

pipe = pipeline(task="text-classification", tokenizer=tokenizer, model=model)

print(pipe("유튜브 내달 2일까지 크리에이터 지원 공간 운영"))

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ./MyBertModel and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[{'label': 'LABEL_1', 'score': 0.5521038770675659}]


In [1]:
# Model 학습
# 1. 토크나이저와 모델 준비
# klue/bert-base 모델과 토크나이저를 사용, 
# KLUE 데이터셋의 기사를 7개 레이블로 분류한 YNAT 테스크 데이터를 학습
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 모델 지정
model_name = "klue/bert-base"

# 미리 학습된 토크나이저/모델을 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)
# 모델은 데이터 클래스 개수에 맞추어 num_labels=7로 설정, 분류 개수 지정
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=7)

/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [2]:
# Model 학습
# 2. 데이터 준비 및 전처리
from datasets import load_dataset

# KLUE 데이터셋 ynat 태스크 데이터 로드, 학습 데이터 45,678건, 검증 데이터 9,107건
dataset = load_dataset("klue", "ynat")
print(f"dataset: {dataset}")

# 각각 컬럼은 'guid', 'title', 'label', 'url', 'date' 5개로 구성
# label: IT과학 0, 경제 1, 사회 2, 생활문화 3, 세계 4, 스포츠 5, 정치 6 총 7개 레이블로 구성
print(set([i for i in dataset["train"]["label"]]))

dataset: DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9107
    })
})
{0, 1, 2, 3, 4, 5, 6}


In [3]:
# Model 학습
# 2. 데이터 준비 및 전처리

# 딕셔너리 형태 입력을 받아서 토큰화 및 인코딩 진행 함수
def tokenize_function(sample):
    result = tokenizer(
        sample["title"],
        padding="max_length", # 뉴스 제목들을 가장 긴 길이에 맞추어 빈 공간을 패딩(padding) 토큰으로 채워준다
    )
    return result

# 작성한 함수를 dataset.map 메서드에 콜백으로 입력하면 전처리 진행
tokenized_dataset = dataset.map(
    function=tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=['guid', 'title', 'url', 'date'],
)
# tokenize_function 함수에서 title 컬럼의 문장들을 가져와 각 단어를 토큰 ID로 쪼개고, 
# 모델이 계산할 수 있는 숫자 꾸러미로 바꾸어 데이터셋에 
# 새로운 컬럼('label(숫자 데이터와 정답 컬럼만 남긴다)', 'input_ids', 'token_type_ids', 'attention_mask')으로 추가
# input_ids 뉴스 제목 문장이 숫자로 쪼개진 토큰 ID 리스트
# attention_mask 어디까지가 실제 단어고 어디부터가 패딩(빈 공간)인지 알려주는 마스크
# token_type_ids 텍스트 1개만 넣을때 0, 텍스트 2개 짝지어 넣을때 1
# label 이 뉴스가 어떤 카테고리(정수 인덱스)인지 나타내는 정답
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9107
    })
})

In [21]:
# Model 학습
# 3. 학습 파라미터 선정
# Trainer를 사용하려면 Accelerate 라이브러리 설치
# 여러개 GPU/서버를 활용해 학습 진행할때 사용, 분산 학습을 진행하기 쉽도록 추상화 기능 제공
# tensorboard 라이브러리 설치, 시각화 사용

# !pip install accelerate -U
# !pip install tensorboard

In [4]:
# Model 학습
# 3. 학습 파라미터 선정
from transformers import Trainer, TrainingArguments, default_data_collator

# TrainingArguments 객체 생성
args = TrainingArguments(
    per_device_train_batch_size=8, # 한번에 64개 뉴스 제목을 묶어서 학습에 투입
    per_device_eval_batch_size=8, # 한번에 64개 뉴스 제목을 묶어서 검증에 투입, 단 평가는 미분을 계산하지 않기 때문에 메모리를 적게 먹는다
    learning_rate=5e-5, # 모델의 가중치를 한번에 얼마나 크게 바꿀것인가 결정, 보통 BERT 계열의 모델은 2e-5(2x10 -5승) ~ 5e-5(5x10 -5승)
    # max_steps=500, # 최대 500번의 스텝(배치 단위 학습)만 하고 학습을 강제 종료(프로토타입 코드 테스트시에 사용됨)
    num_train_epochs=3,
    evaluation_strategy="steps",
    logging_strategy="steps",
    logging_steps=50,
    logging_dir="./content/logs",
    save_steps=50,
    output_dir="./content/ckpt",
    report_to="tensorboard", # 시각화 도구(텐서보드) 연동, 실시간으로 학습과정(손실/정확도) 그래프를 볼 수 있도록 설정
)

# Trainer 객체 생성
trainer = Trainer(
    model=model, # 학습 모델(klue/bert-base, 뉴스 카테고리 7개 분류용으로 설정된 모델), 이 모델의 가중치(weight)를 데이터에 맞게 업데이트 지정
    args=args, # TrainingArguments 객체를 통한 학습 설계도(배치 크기, 학습률, 최대 스텝, 저장 경로...등)
    train_dataset=tokenized_dataset["train"].select(range(500)), # 학습용 데이터셋(모델이 실제로 읽고 학습하면서 가중치를 수정해 나갈 정답과 토큰이 들어있는 훈령요 데이터)
    eval_dataset=tokenized_dataset["validation"].select(range(100)), # 검증 데이터셋(50 스텝마다 모델이 학습을 잠깐 멈추고 평가 및 로깅)
    tokenizer=tokenizer, # 텍스트롤 숫자로 쪼개는 토크나이저
    data_collator=default_data_collator, # 데이터를 텐서 배치로 묶음(Batch)을 조립: ['label', 'input_ids', 'token_type_ids', 'attention_mask'] 
)

/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/lib/python3.11/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [23]:
# Model 학습
# 4. 학습 진행
# 생성한 Trainer 인스턴스로 학습을 진행
trainer.train()

  0%|          | 0/189 [00:00<?, ?it/s]

{'loss': 1.3135, 'grad_norm': 9.722713470458984, 'learning_rate': 3.677248677248677e-05, 'epoch': 0.79}


  0%|          | 0/13 [00:00<?, ?it/s]

{'eval_loss': 0.7272064089775085, 'eval_runtime': 65.8753, 'eval_samples_per_second': 1.518, 'eval_steps_per_second': 0.197, 'epoch': 0.79}
{'loss': 0.4462, 'grad_norm': 10.98563003540039, 'learning_rate': 2.3544973544973546e-05, 'epoch': 1.59}


  0%|          | 0/13 [00:00<?, ?it/s]

{'eval_loss': 0.6371586322784424, 'eval_runtime': 66.1316, 'eval_samples_per_second': 1.512, 'eval_steps_per_second': 0.197, 'epoch': 1.59}
{'loss': 0.2113, 'grad_norm': 0.28513458371162415, 'learning_rate': 1.0317460317460318e-05, 'epoch': 2.38}


  0%|          | 0/13 [00:00<?, ?it/s]

{'eval_loss': 0.7128219604492188, 'eval_runtime': 65.0412, 'eval_samples_per_second': 1.537, 'eval_steps_per_second': 0.2, 'epoch': 2.38}
{'train_runtime': 6460.2329, 'train_samples_per_second': 0.232, 'train_steps_per_second': 0.029, 'train_loss': 0.5454031903907736, 'epoch': 3.0}


TrainOutput(global_step=189, training_loss=0.5454031903907736, metrics={'train_runtime': 6460.2329, 'train_samples_per_second': 0.232, 'train_steps_per_second': 0.029, 'total_flos': 394684300800000.0, 'train_loss': 0.5454031903907736, 'epoch': 3.0})

In [11]:
# Model 학습
# 5. 성능 평가
# 텐서보드로 시각화, Trainer에서는 손실을 수치로만 보여주는데 그래프로 확인하는 것이 더 직관적이므로 시각화가 필요함
# 웹 화면에서 확인: http://localhost:6007

# 주피터 노트북 환경에서 텐서보드(TensorBoard) 확장 프로그램을 로드
%load_ext tensorboard

# 로컬에 저장된 로그 파일을 읽어와 웹 대시보드(그래프창)를 실행
%tensorboard --logdir ./content/logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6007 (pid 9548), started 0:11:04 ago. (Use '!kill 9548' to kill it.)

In [ ]:
# Model 학습
# 6. 모델 저장 
output_dir = "./content/drive/MyDrive/trained_model"
trainer.save_model(output_dir)

In [ ]:
# Model 학습
# 7. 모델 로드
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 사전 학습된 모델 로드: 여기서는 파인튜닝 학습한 모델을 로컬에서 불러옴
tokenizer = AutoTokenizer.from_pretrained(output_dir)
model = AutoModelForSequenceClassification.from_pretrained(output_dir)

print(f"tokenizer: {tokenizer}")
print(f"model: {model}")

# 모델 정보: BertTokenizer 모델
# vocab_size=32000 이 모델이 아는 총 단어(어휘) 개수가 32,000개, 
# model_max_length=512 모델이 한번에 처리할 수 있는 최대 글자(토큰) 길이 제한이 512개
# tokenizer: BertTokenizerFast(name_or_path='./content/drive/MyDrive/trained_model', vocab_size=32000, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
# 	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
# 	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
# 	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
# 	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
# 	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
# }

# 모델 정보: BertForSequenceClassification 모델
# Embedding(32000, 768, padding_idx=0) 32,000개의 단어 각각을 768차원의 벡터(숫자 공간)로 표현
# model: BertForSequenceClassification(
#   (bert): BertModel(
#     (embeddings): BertEmbeddings(
#       (word_embeddings): Embedding(32000, 768, padding_idx=0)
#       (position_embeddings): Embedding(512, 768)
#       (token_type_embeddings): Embedding(2, 768)
#       (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
#       (dropout): Dropout(p=0.1, inplace=False)
#     )

# ModuleList (0-11): 12 x BertLayer BERT의 핵심인 트랜스포머 레이어가 총 12층(Layer)으로 겹겹이 쌓여 텍스트의 맥락을 깊이 있게 이해한다
#     (encoder): BertEncoder(
#       (layer): ModuleList(
#         (0-11): 12 x BertLayer(
#           (attention): BertAttention(
#             (self): BertSdpaSelfAttention(
#               (query): Linear(in_features=768, out_features=768, bias=True)
#               (key): Linear(in_features=768, out_features=768, bias=True)
#               (value): Linear(in_features=768, out_features=768, bias=True)
#               (dropout): Dropout(p=0.1, inplace=False)
#             )
#             (output): BertSelfOutput(
#               (dense): Linear(in_features=768, out_features=768, bias=True)
#               (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
#               (dropout): Dropout(p=0.1, inplace=False)
#             )
#           )
#           (intermediate): BertIntermediate(
#             (dense): Linear(in_features=768, out_features=3072, bias=True)
#             (intermediate_act_fn): GELUActivation()
#           )
#           (output): BertOutput(
#             (dense): Linear(in_features=3072, out_features=768, bias=True)
#             (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
#             (dropout): Dropout(p=0.1, inplace=False)
#           )
#         )
#       )
#     )
#     (pooler): BertPooler(
#       (dense): Linear(in_features=768, out_features=768, bias=True)
#       (activation): Tanh()
#     )
#   )
#   (dropout): Dropout(p=0.1, inplace=False)

# (classifier): Linear(in_features=768, out_features=7, bias=True) 마지막 768차원의 특징을 받아 7개의 카테고리(출력 7개)
# 어디에 속하는지 점수를 매겨주는 최종 분류기(Linear 레이어)
#   (classifier): Linear(in_features=768, out_features=7, bias=True)
# )

tokenizer: BertTokenizerFast(name_or_path='./content/drive/MyDrive/trained_model', vocab_size=32000, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
model: BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word

In [ ]:
# 허깅페이스 허브 등록
# 1. push_to_hub()
# 직접 미세조정한 모델을 push_to_hub 메서드를 이용하여 업로드
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from huggingface_hub import login, delete_repo, create_repo # 코드로 로그인하는 함수 임포트

# 모델 지정
model_name = "./content/drive/MyDrive/trained_model"

# 사전 학습 모델
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 리포지터리 이름 지정
USERNAME = "CHY760MARIN"
REPO_NAME = f"{USERNAME}/my-bert-fine-tuned1"
# 허깅페이스 토큰
AUTH_TOKEN = ""

# # 리포지터리 이름 지정
# USERNAME = ""
# REPO_NAME = ""
# # 허깅페이스 토큰
# AUTH_TOKEN = ""

# 허깅페이스 허브 등록: model
model.push_to_hub(
    REPO_NAME, # 리포지터리 이름
    use_temp_dir=True,
    token=AUTH_TOKEN, # 허깅페이스 토큰
)
# 허깅페이스 허브 등록: tokenizer
tokenizer.push_to_hub(
    REPO_NAME, # 리포지터리 이름
    use_temp_dir=True,
    token=AUTH_TOKEN, # 허깅페이스 토큰
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/CHY760MARIN/my-bert-fine-tuned1/commit/14e212b861dfc51987b38e858bc34ce1aa5febfa', commit_message='Upload tokenizer', commit_description='', oid='14e212b861dfc51987b38e858bc34ce1aa5febfa', pr_url=None, repo_url=RepoUrl('https://huggingface.co/CHY760MARIN/my-bert-fine-tuned1', endpoint='https://huggingface.co', repo_type='model', repo_id='CHY760MARIN/my-bert-fine-tuned1'), pr_revision=None, pr_num=None)

In [ ]:
# 허깅페이스 허브 등록
# 2. 허깅페이스 리포지터리 생성
from huggingface_hub import login, delete_repo, create_repo # 코드로 로그인하는 함수 임포트

# 로그인
login(token=AUTH_TOKEN)

# 리포지터리 이름 지정
USERNAME = "CHY760MARIN"
REPO_NAME = f"{USERNAME}/my-bert-fine-tuned2"

# 허깅페이스 리포지터리 생성(이미 존재해도 에러 나지 않도록 exist_ok=True 설정)
create_repo(repo_id=REPO_NAME, exist_ok=True)

print(f"'{REPO_NAME}', 리포지터리 생성 완료!")

In [28]:
# 허깅페이스 허브 등록
# 3. 허깅페이스 리포지터리 삭제
from huggingface_hub import login, delete_repo, create_repo # 코드로 로그인하는 함수 임포트

# 로그인
login(token=AUTH_TOKEN)

# 리포지터리 이름 지정
USERNAME = "CHY760MARIN"
REPO_NAME = f"{USERNAME}/my-bert-fine-tuned2"

# 허깅페이스 리포지터리 삭제
delete_repo(
    repo_id=REPO_NAME,
    repo_type="model" # 모델 리포지터리인 경우 "model" 지정 (기본값도 "model")
)

print(f"'{REPO_NAME}' 리포지터리가 성공적으로 삭제되었습니다!")

'CHY760MARIN/my-bert-fine-tuned2' 리포지터리가 성공적으로 삭제되었습니다!


In [26]:
# 허깅페이스 허브 등록
# 4. 허깅페이스 모델 업로드
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 모델 지정
model_name = "./content/drive/MyDrive/trained_model"

# 사전 학습 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 리포지터리 이름 지정
USERNAME = "CHY760MARIN"
REPO_NAME = f"{USERNAME}/my-bert-fine-tuned2"

# 모델 업로드
print("모델 업로드 중...")

tokenizer.push_to_hub(
    REPO_NAME, # 리포지터리 이름
    use_temp_dir=True, # 임시 폴더에 먼저 모아서 업로드한 뒤 업로드가 끝나면 삭제
    token=AUTH_TOKEN, # 허깅페이스 토큰
)
model.push_to_hub(
    REPO_NAME, # 리포지터리 이름
    use_temp_dir=True, # 임시 폴더에 먼저 모아서 업로드한 뒤 업로드가 끝나면 삭제
    token=AUTH_TOKEN, # 허깅페이스 토큰
)

print("업로드 완료!")

모델 업로드 중...


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

업로드 완료!


In [ ]:
# 허깅페이스 허브 등록
# 5. 허깅페이스 허브 다운로드 테스트
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 모델 이름 지정
USERNAME = "CHY760MARIN"
model_name = f"{USERNAME}/my-bert-fine-tuned1"

# 사전 학습 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# 허깅페이스 허브 모델 확인
print(f"tokenizer: {tokenizer}")
print(f"model: {model}")

tokenizer: BertTokenizerFast(name_or_path='CHY760MARIN/my-bert-fine-tuned1', vocab_size=32000, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
model: BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embed

In [ ]:
# Tokenizer 학습
# 1. KLUE/YNAT 데이터셋 로드 단계
from datasets import load_dataset

# KLUE/YNAT 데이터셋 로드
dataset = load_dataset("klue", "ynat")

# Train 데이터셋 확인
dataset["train"][0]

{'guid': 'ynat-v1_train_00000',
 'title': '유튜브 내달 2일까지 크리에이터 지원 공간 운영',
 'label': 3,
 'url': 'https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=105&sid2=227&oid=001&aid=0008508947',
 'date': '2016.06.30. 오전 10:36'}

In [8]:
# Tokenizer 학습
# 2. KLUE/YNAT 데이터셋 전처리 단계

# 파일 저장 경로
target_key = "title"
file_path = f"./content/tokenizer_data_{target_key}.txt"

# 모든 train, validation의 title을 하나의 리스트에 모은다
all_titles = []
# dataset.keys(): train, validation 의 데이터셋의 분할 이름 구성
for split_name in dataset.keys():
    # 객체(리스트 등..)을 넣으면 그안의 원소들을 하나씩 낱개로 꺼내어(Unpacking) 기존 리스트의 끝에 쭉 이어 붙인다
    all_titles.extend(dataset[split_name][target_key])

# 한번의 파일로 저장
with open(file_path, "w", encoding="utf-8") as f:
    # 리스트에서 한개씩 데이터를 꺼내서 줄바꿈을 붙여준다
    # '첫 번째 뉴스' + \n + '두 번째 뉴스' + \n + '세 번째 뉴스'
    f.write("\n".join(all_titles))

# 데이터 확인
print(f"총 {len(all_titles)}개의 타이틀의 {file_path}에 저장되었습니다.")

총 54785개의 타이틀의 ./content/tokenizer_data_title.txt에 저장되었습니다.


In [ ]:
# Tokenizer 학습
# 3. 특수 토큰 정의

# 모델이 특별하게 취급해야하는 단어들(특수토큰)을 미리 목록으로 정의
user_defined_symbols = [
    "[PAD]", # 문장 길이를 맞추기 위해 사용되는 토큰
    "[UNK]", # 토크나이저가 인식할 수 없는 토큰
    "[CLS]", # BERT 계열 모델에서 문장 전체 정보를 저장하는 토큰
    "[SEP]", # BERT 계열 모델에서 문장 구분을 위해 사용되는 토큰
    "[MASK]" # Masked LM에서 토큰 마스킹을 위해 사용하는 토큰
]

# 구글 BERT 모델 사전을 만들때 전통: [UNUSED0] 부터 [UNUSED99] 까지 정확히 100개를 기본
# 파인 튜닝시 일반적인 단어 외에 새로운 특수 기호나 태그를 모델에 억지로 주입하는 상황 발생실
# 예) 특정 도메인(의학,법률..등)에서 중요한 전문 기호나, 대화 모델에서 화자 분리 구분하는 태그(<user>,<bot>) 추가시
# 실무에서 모델을 쓸때 보통 수십개 내외의 커스텀 특수토큰을 활용하게 되며 통상적으로 100개를 기본으로 사용
# 예비토큰 100개 생성: 여기는 나중에 쓸 빈 상자(예비석), 나중에 새로운 토큰을 넣을때 이 빈자리를 재활용 한다
unused_token_num = 100
unused_list = [f"[UNUSED{i}]" for i in range(100)]

# 전체 특수 토큰 합치기: 특수토큰 5개 + 예비토큰 100개를 하나의 리스트로 합쳐서 최종 완성
whole_user_defined_symbols = user_defined_symbols + unused_list

print(whole_user_defined_symbols)

['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '[UNUSED0]', '[UNUSED1]', '[UNUSED2]', '[UNUSED3]', '[UNUSED4]', '[UNUSED5]', '[UNUSED6]', '[UNUSED7]', '[UNUSED8]', '[UNUSED9]', '[UNUSED10]', '[UNUSED11]', '[UNUSED12]', '[UNUSED13]', '[UNUSED14]', '[UNUSED15]', '[UNUSED16]', '[UNUSED17]', '[UNUSED18]', '[UNUSED19]', '[UNUSED20]', '[UNUSED21]', '[UNUSED22]', '[UNUSED23]', '[UNUSED24]', '[UNUSED25]', '[UNUSED26]', '[UNUSED27]', '[UNUSED28]', '[UNUSED29]', '[UNUSED30]', '[UNUSED31]', '[UNUSED32]', '[UNUSED33]', '[UNUSED34]', '[UNUSED35]', '[UNUSED36]', '[UNUSED37]', '[UNUSED38]', '[UNUSED39]', '[UNUSED40]', '[UNUSED41]', '[UNUSED42]', '[UNUSED43]', '[UNUSED44]', '[UNUSED45]', '[UNUSED46]', '[UNUSED47]', '[UNUSED48]', '[UNUSED49]', '[UNUSED50]', '[UNUSED51]', '[UNUSED52]', '[UNUSED53]', '[UNUSED54]', '[UNUSED55]', '[UNUSED56]', '[UNUSED57]', '[UNUSED58]', '[UNUSED59]', '[UNUSED60]', '[UNUSED61]', '[UNUSED62]', '[UNUSED63]', '[UNUSED64]', '[UNUSED65]', '[UNUSED66]', '[UNUSED67]', '[UNUSED68]

In [ ]:
# Tokenizer 학습
# 4. 토크나이저의 WordPiece 슬롯에 장착하기
from tokenizers import Tokenizer # 토크나이저 기본 베이스 import
from tokenizers.models import WordPiece # WordPiece 클래스 import

# 토크나이저에 WordPiece는 텍스트를 글자 조각(Subword) 단위로 쪼개는 알고리즘 장작하기
# WordPiece(unk_token="[UNK]") 만약 사전에 없는 완전히 새로운 단어나 외계어 -> [UNK] 토큰으로 처리
# Tokenizer(WordPiece(unk_token="[UNK]")) WordPiece 규칙을 메인 Tokenizer 컨테이너 안에 넣어 
# bert_tokenizer 실제 사용할 토크나이저 객체 생성
bert_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))

In [ ]:
# Tokenizer 학습
# 5. 토크나이저의 normalizer 슬롯에 장착하기
from tokenizers import normalizers # 정규화 모듈 가져오기

# BERT 표준 정규화 객체 생성
normalizer = normalizers.BertNormalizer()

# 토크나이저에 정규화 규칙 장착하기
bert_tokenizer.normalizer = normalizer

# 결과 확인
normalizer.normalize_str("HeLLo  WOrld!  It’s   a  café   with  accénts  &amp;   weird   spaces   and   FULL-WIDTH   ｃｈａｒｓ.")
# 'hello  world!  it’s   a  cafe   with  accents  &amp;   weird   spaces   and   full-width   ｃｈａｒｓ.'

'hello  world!  it’s   a  cafe   with  accents  &amp;   weird   spaces   and   full-width   ｃｈａｒｓ.'

In [ ]:
# Tokenizer 학습
# 6. 토크나이저의 pre_tokenizer 슬롯에 장착하기
from tokenizers.pre_tokenizers import Whitespace # Whitespace 불러오기

# Whitespace 줄바꿈/공백 단위로 구분하여 잘라내는 토크나이저 객체 생성
pre_tokenizer = Whitespace()

# 토크나이저의 pre_tokenizer 슬롯에 장착하기
bert_tokenizer.pre_tokenizer = pre_tokenizer

# 결과 확인
pre_tokenizer.pre_tokenize_str("안녕하세요. 제대로 인코딩이 되는지 확인 중입니다.")

[('안녕하세요', (0, 5)),
 ('.', (5, 6)),
 ('제대로', (7, 10)),
 ('인코딩이', (11, 15)),
 ('되는지', (16, 19)),
 ('확인', (20, 22)),
 ('중입니다', (23, 27)),
 ('.', (27, 28))]

In [16]:
# Tokenizer 학습
# 7. 토크나이저의 post_processor 슬롯에 장착하기
from tokenizers.processors import TemplateProcessing # TemplateProcessing 불러오기

# BERT 전용 템플릿 양식 규칙 정의하기
# BERT 모델은 문장 가장 앞에 [CLS] 토큰이 있어야 하고, 두 문장을 입력받았을때 문장을 구별하기 위한 [SEP] 토큰으로 감싸져 있어야 한다
post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]", # 단일 문장일때 [CLS] 문장 [SEP] 형태
    pair="[CLS] $A [SEP] $B:1 [SEP]:1", # 두 문장이 쌍 일때 [CLS] 문장1 [SEP] 문장2 [SEP] 형태
    special_tokens=[(t, i) for i, t in enumerate(user_defined_symbols)],
)

# 토크나이저의 post_processor 슬롯에 장착하기
bert_tokenizer.post_processor = post_processor

In [ ]:
# Tokenizer 학습
# 8. 조립된 토크나이저 학습 설정
from tokenizers.trainers import WordPieceTrainer # WordPieceTrainer 불러오기

# 준비된 텍스트 파일(코퍼스)를 보고, 가장 많이 쓰이는 빈도수 기반으로 단어 사전(vocabulary)를
# 어떻게 만들어낼지 규칙을 지시하는 학습 설정 단계
vocab_size = 24000
trainer = WordPieceTrainer(
    vocab_size=vocab_size, # 단어사전 크기 2400 지정
    special_tokens=whole_user_defined_symbols, # 전체 특수토큰
)

In [18]:
# Tokenizer 학습
# 9. 조립된 토크나이저 학습 진행
from glob import glob # glob(파일 경로 찾기) 불러오기

# 토크나이저 학습 진행
# files=glob(f"./content/*.txt") 폴더 안에서 특정 패턴을 가진 파일들을 가져온다 예).txt 파일들을 한꺼번에 모두 가져옴
# trainer=trainer 내부적으로 트레이너가 텍스트 파일들을 처음부터 끝까지 흝으면서 단어와 조각(Subword)들의 등장 빈도를 통계를 낸다
# 빈도가 높은 순서대로 차곡차곡 쌓아 올리되, 우리가 지정한 최대 사전 크기인 24,000개가 될때까지 단어를 선별한다
bert_tokenizer.train(files=glob(f"./content/*.txt"), trainer=trainer)

In [ ]:
# Tokenizer 학습
# 10. 인코딩 후 다시 디코딩 진행해 학습 결과 확인
# 토크나이저 슬롯에 디코더가 없는 상태이며, 이와 같이 어색한 디코딩 결과를 확인 할 수 있다
output_encoded = bert_tokenizer.encode("인코딩 및 디코딩이 제대로 이루어지는지 확인 중입니다.")

# 인코딩 결과 확인
print(f"output_encoded: {output_encoded.ids}")

# 디코딩 결과 확인
output_decoded = bert_tokenizer.decode(output_encoded.ids)
print(f"output_decoded: {output_decoded}")

output_encoded: [2, 675, 906, 2220, 4518, 1240, 906, 2220, 569, 6727, 12916, 10780, 586, 1881, 16617, 10189, 106, 3]
output_decoded: 인 ##코 ##딩 및 디 ##코 ##딩 ##이 제대로 이루 ##어지는 ##지 확인 중이 ##ᆸ니다 .


In [21]:
# Tokenizer 학습
# 11. 학습한 토크나이저 디코더에 Wordpiece 디코더 할당
from tokenizers import decoders

# 토크나이저의 decoder 슬롯에 장착하기
bert_tokenizer.decoder = decoders.WordPiece()

bert_tokenizer.decode(output_encoded.ids)

'인코딩 및 디코딩이 제대로 이루어지는지 확인 중입니다.'

In [ ]:
# Tokenizer 학습
# 12. 목표: Transfomers 라이브러리에서 사용되는 토크나이저를 만드는 것
from transformers import BertTokenizerFast # BertTokenizerFast 불러오기

# 지금까지 조립하고 학습시킨 순수 토크나이저를 허깅페이스의 메인 모델 학습 라이브러리(transformers)에서
# 곧바로 불러다 쓸 수 있는 공식 BERT 전용 패스트 토크나이저 형태로 포장하는 단계

# 지금까지 만든 순수 토크나이저를 트랜스포머스 인자에 넣어, 동시에 허깅페이스 트랜스포머스 생태계에세 완변하게 호환
fast_tokenizer = BertTokenizerFast(tokenizer_object=bert_tokenizer)
encoded = fast_tokenizer.encode("인코딩 및 디코딩이 제대로 이루어지는지 확인 중입니다.")
decoded = fast_tokenizer.decode(encoded)

# 결과 확인
print(encoded)
print(decoded)

[2, 675, 906, 2220, 4518, 1240, 906, 2220, 569, 6727, 12916, 10780, 586, 1881, 16617, 10189, 106, 3]
[CLS] 인코딩 및 디코딩이 제대로 이루어지는지 확인 중입니다. [SEP]


/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [1]:
# Tokenizer 학습
# 13. Transformers 라이브러이에서 사용되는 형식대로 모델 저장
output_dir = "./content/MyTokenizer"
fast_tokenizer.save_pretrained(output_dir)

NameError: name 'fast_tokenizer' is not defined

In [6]:
# Tokenizer 학습
# 14. Tokenizer 모델 로드 및 확인
from transformers import BertTokenizerFast

# 지금까지 조립하고 학습시킨 순수 토크나이저를 허깅페이스의 메인 모델 학습 라이브러리(transformers)에서
# 곧바로 불러다 쓸 수 있는 공식 BERT 전용 패스트 토크나이저 형태로 포장하는 단계
# 지금까지 만든 순수 토크나이저를 트랜스포머스 인자에 넣어, 동시에 허깅페이스 트랜스포머스 생태계에세 완변하게 호환
output_dir = "./content/MyTokenizer"
new_tokenizer = BertTokenizerFast.from_pretrained(output_dir)

encoded = new_tokenizer(["인코딩 잘 되는지 확인", "안되면 다시 학습하자"])

for k,v in encoded.items():
    print(k, v)

print(new_tokenizer.decode(encoded["input_ids"][0]))
print(new_tokenizer.decode(encoded["input_ids"][1]))

input_ids [[2, 675, 906, 2220, 1675, 6464, 586, 1881, 3], [2, 18632, 1594, 6985, 3782, 3]]
token_type_ids [[0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]]
attention_mask [[1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]
[CLS] 인코딩 잘 되는지 확인 [SEP]
[CLS] 안되면 다시 학습하자 [SEP]


In [55]:
# Tokenizer 학습
# 15. 모델 초기화 후 학습
# - BERT 모델을 초기화하여 처음부터 학습
from datasets import load_dataset
from transformers import BertTokenizerFast
from transformers import BertForMaskedLM
from transformers import BertConfig

# KLUE YNAT 데이터셋 로드
dataset = load_dataset("klue", "ynat")

# 모델 이름 지정
model_name = "./content/MyTokenizer"
# 사전 학습 모델 로드
tokenizer = BertTokenizerFast.from_pretrained(model_name)

# 모델의 최초 선언을 위해서 해당 모델 BertConfig 선언
cfg = BertConfig
print(f"BertConfig: {cfg}")

# 모델 BertConfig 수정, 모델 vocab_size는 tokenizer의 vocab_size를 따라가야 하기에 내용을 변경하여 다시 선언
mycfg = BertConfig(vocab_size=tokenizer.vocab_size)
print(f"BertConfig 커스텀: {mycfg}")

# BertForMaskedLM 마스킹된 언어 모델은 문장 토큰 일부(약 15%)를 [MASK] 토큰으로 바꾸고 바뀐 원래 토큰을 추론
model = BertForMaskedLM(mycfg)
print(f"BertForMaskedLM: {model.config}")

loading file vocab.txt
loading file tokenizer.json
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
Generate config GenerationConfig {
  "pad_token_id": 0
}



BertConfig: <class 'transformers.models.bert.configuration_bert.BertConfig'>
BertConfig 커스텀: BertConfig {
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.44.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 24000
}

BertForMaskedLM: BertConfig {
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_t

In [56]:
# Tokenizer 학습
# 15. 모델 초기화 후 학습
# - 데이터 준비 및 전처리

# 딕셔너리 형태 입력을 받아서 토큰화 및 인코딩 진행 함수
def tokenize_function(sample):
    result = tokenizer(
        sample["title"],
        padding="max_length", # 뉴스 제목들을 가장 긴 길이에 맞추어 빈 공간을 패딩(padding) 토큰으로 채워준다
        truncation=True, # 최대 길이 128번째 토큰에서 잘라냄(Truncation)
        max_length=128, # 최대 길이를 명시
    )
    return result

# 작성한 함수를 dataset.map 메서드에 콜백으로 입력하면 전처리 진행
tokenized_dataset = dataset.map(
    function=tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=['guid', 'title', 'url', 'date', 'label'],
)
# tokenize_function 함수에서 title 컬럼의 문장들을 가져와 각 단어를 토큰 ID로 쪼개고, 
# 모델이 계산할 수 있는 숫자 꾸러미로 바꾸어 데이터셋에 
# 새로운 컬럼('label(숫자 데이터와 정답 컬럼만 남긴다)', 'input_ids', 'token_type_ids', 'attention_mask')으로 추가
# input_ids 뉴스 제목 문장이 숫자로 쪼개진 토큰 ID 리스트
# attention_mask 어디까지가 실제 단어고 어디부터가 패딩(빈 공간)인지 알려주는 마스크
# token_type_ids 텍스트 1개만 넣을때 0, 텍스트 2개 짝지어 넣을때 1
# label 이 뉴스가 어떤 카테고리(정수 인덱스)인지 나타내는 정답
tokenized_dataset

Map:   0%|          | 0/9107 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9107
    })
})

In [57]:
# Tokenizer 학습
# 15. 모델 초기화 후 학습
# - 학습 파라미터 선정
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# 마스킹 전용(mlm=True로 설정하면 마스크드 언어 모델용으로 동작)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15 # 문장 속 단어 15%를 무작위로 골라 [MASK] 토큰으로 마스킹
)

# TrainingArguments 객체 생성, 학습 하이퍼파라미터 및 환경 설정
args = TrainingArguments(
    per_device_train_batch_size=16, # 한번에 64개 뉴스 제목을 묶어서 학습에 투입
    per_device_eval_batch_size=16, # 한번에 64개 뉴스 제목을 묶어서 검증에 투입, 단 평가는 미분을 계산하지 않기 때문에 메모리를 적게 먹는다
    learning_rate=5e-5, # 모델의 가중치를 한번에 얼마나 크게 바꿀것인가 결정, 보통 BERT 계열의 모델은 2e-5(2x10 -5승) ~ 5e-5(5x10 -5승)
    # max_steps=500, # 최대 500번의 스텝(배치 단위 학습)만 하고 학습을 강제 종료(프로토타입 코드 테스트시에 사용됨)
    num_train_epochs=3, # 전체 학습 데이터를 총 3번 반복해서 학습
    evaluation_strategy="steps",
    logging_strategy="steps",
    logging_steps=50, # 50번 학습 스텝이 돌때마다 현재 상태를 로그 경로에 기록 및 체크포인트를 저장
    logging_dir="./content/logs", # 학습 과정의 손실(Loss) 변화는 텐서보드로 보며, 경로는 해당 경로
    save_steps=50, # 50번 학습 스텝
    output_dir="./content/bert_masked_lm_ckpt", # 모델 가중치 저장 경로
    report_to="tensorboard", # 시각화 도구(텐서보드) 연동, 실시간으로 학습과정(손실/정확도) 그래프를 볼 수 있도록 설정
)

# Trainer 객체 생성, 학습 총괄 매니저 역할
trainer = Trainer(
    model=model, # 학습 모델(klue/bert-base, 뉴스 카테고리 7개 분류용으로 설정된 모델), 이 모델의 가중치(weight)를 데이터에 맞게 업데이트 지정
    args=args, # TrainingArguments 객체를 통한 학습 설계도(배치 크기, 학습률, 최대 스텝, 저장 경로...등)
    train_dataset=tokenized_dataset["train"].select(range(3000)), # 학습용 데이터셋(모델이 실제로 읽고 학습하면서 가중치를 수정해 나갈 정답과 토큰이 들어있는 훈령요 데이터)
    eval_dataset=tokenized_dataset["validation"].select(range(600)), # 검증 데이터셋(50 스텝마다 모델이 학습을 잠깐 멈추고 평가 및 로깅)
    tokenizer=tokenizer, # 텍스트롤 숫자로 쪼개는 토크나이저
    data_collator=data_collator, # 예) 배치 8개의 문장중 첫번째 문장을 꺼내서 그중에 15% 마스킹
)

/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/lib/python3.11/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
using `logging_steps` to initialize `eval_steps` to 50
PyTorch: setting up devices


In [58]:
# Tokenizer 학습
# 15. 모델 초기화 후 학습
# - 학습 진행

# 생성한 Trainer 인스턴스로 학습을 진행
trainer.train()

***** Running training *****
  Num examples = 3,000
  Num Epochs = 3
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 564
  Number of trainable parameters = 104,498,880


  0%|          | 0/564 [00:00<?, ?it/s]


***** Running Evaluation *****
  Num examples = 600
  Batch size = 16


{'loss': 9.7463, 'grad_norm': 6.892976760864258, 'learning_rate': 4.556737588652483e-05, 'epoch': 0.27}


  0%|          | 0/38 [00:00<?, ?it/s]

...

Saving model checkpoint to ./content/bert_masked_lm_ckpt/checkpoint-550
Configuration saved in ./content/bert_masked_lm_ckpt/checkpoint-550/config.json
Configuration saved in ./content/bert_masked_lm_ckpt/checkpoint-550/generation_config.json


{'eval_loss': 9.437905311584473, 'eval_runtime': 111.0649, 'eval_samples_per_second': 5.402, 'eval_steps_per_second': 0.342, 'epoch': 2.93}


Model weights saved in ./content/bert_masked_lm_ckpt/checkpoint-550/model.safetensors
tokenizer config file saved in ./content/bert_masked_lm_ckpt/checkpoint-550/tokenizer_config.json
Special tokens file saved in ./content/bert_masked_lm_ckpt/checkpoint-550/special_tokens_map.json
Saving model checkpoint to ./content/bert_masked_lm_ckpt/checkpoint-564
Configuration saved in ./content/bert_masked_lm_ckpt/checkpoint-564/config.json
Configuration saved in ./content/bert_masked_lm_ckpt/checkpoint-564/generation_config.json
Model weights saved in ./content/bert_masked_lm_ckpt/checkpoint-564/model.safetensors
tokenizer config file saved in ./content/bert_masked_lm_ckpt/checkpoint-564/tokenizer_config.json
Special tokens file saved in ./content/bert_masked_lm_ckpt/checkpoint-564/special_tokens_map.json


Training completed. Do not forget to share your model on huggingface.co/models =)




{'train_runtime': 7891.0556, 'train_samples_per_second': 1.141, 'train_steps_per_second': 0.071, 'train_loss': 9.211995862054486, 'epoch': 3.0}


TrainOutput(global_step=564, training_loss=9.211995862054486, metrics={'train_runtime': 7891.0556, 'train_samples_per_second': 1.141, 'train_steps_per_second': 0.071, 'total_flos': 592165748736000.0, 'train_loss': 9.211995862054486, 'epoch': 3.0})

In [59]:
# Tokenizer 학습
# 15. 모델 초기화 후 학습
# - 성능 평가
# 텐서보드로 시각화, Trainer에서는 손실을 수치로만 보여주는데 그래프로 확인하는 것이 더 직관적이므로 시각화가 필요함
# 웹 화면에서 확인: http://localhost:6007

# 주피터 노트북 환경에서 텐서보드(TensorBoard) 확장 프로그램을 로드
%load_ext tensorboard

# 로컬에 저장된 로그 파일을 읽어와 웹 대시보드(그래프창)를 실행
%tensorboard --logdir ./content/logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 12421), started 9:30:10 ago. (Use '!kill 12421' to kill it.)

In [60]:
# Tokenizer 학습
# 15. 모델 초기화 후 학습
# - 모델 저장 
save_path = "./content/drive/MyDrive/trained_bert_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"모델이 안전하게 저장되었습니다: {save_path}")

Saving model checkpoint to ./content/drive/MyDrive/trained_bert_model
Configuration saved in ./content/drive/MyDrive/trained_bert_model/config.json
Configuration saved in ./content/drive/MyDrive/trained_bert_model/generation_config.json
Model weights saved in ./content/drive/MyDrive/trained_bert_model/model.safetensors
tokenizer config file saved in ./content/drive/MyDrive/trained_bert_model/tokenizer_config.json
Special tokens file saved in ./content/drive/MyDrive/trained_bert_model/special_tokens_map.json
tokenizer config file saved in ./content/drive/MyDrive/trained_bert_model/tokenizer_config.json
Special tokens file saved in ./content/drive/MyDrive/trained_bert_model/special_tokens_map.json


모델이 안전하게 저장되었습니다: ./content/drive/MyDrive/trained_bert_model


In [61]:
# Tokenizer 학습
# 16. 마스크드 언어 모델(MLM) 추론 테스트 코드
# - pipeline 추상화 코드

# Pipeline: 토크나이저 생성, 텐서 변환, 모델 추론, 디코딩 등의 복잡한 과정을 알아서 내부적으로 싹 감싸서(추상화하여) 처리해 주는 편리한 도구
# 허깅페이스 허브 모델 가져오기
from transformers import pipeline

# 1. 방금 학습하고 저장한 체크포인트 경로에서 모델과 토크나이저 불러오기
model_path = "./content/bert_masked_lm_ckpt/checkpoint-564" # 가장 마지막 체크포인트 경로

# pipeline 설정
fill_mask = pipeline(
    task="fill-mask",
    model=model_path,
    tokenizer=model_path,
)

# 2. 테스트 문장 작성([MASK] 토큰 포함)
# KLUE/YNAT 뉴스 데이터셋 도메인에 맞는 문장이나 임의의 문장
text = "오늘 서울에서 [MASK]가 개최되었다."

# 3. 결과 예측 수행
results = fill_mask(inputs=text)

# 4. 결과 출력
print(f"입력 문장: {text}\n")
print(f"모델 예측 결과:")
for i, res in enumerate(results):
    # print(f"{i}. {res}")
    print(f"{i+1}. 토큰: {res['token_str']} (점수: {res['score']:.4f}) -> 문장 완성: {res['sequence']}")

loading configuration file ./content/bert_masked_lm_ckpt/checkpoint-564/config.json
Model config BertConfig {
  "_name_or_path": "./content/bert_masked_lm_ckpt/checkpoint-564",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.44.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 24000
}

loading configuration file ./content/bert_masked_lm_ckpt/checkpoint-564/config.json
Model config BertConfig {
  "_name_or_path": "./content/bert_masked_lm_ckpt/checkpoint-564",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attenti

입력 문장: 오늘 서울에서 [MASK]가 개최되었다.

모델 예측 결과:
1. 토큰: … (점수: 0.0360) -> 문장 완성: 오늘 서울에서 … 가 개최되었다.
2. 토큰: · (점수: 0.0130) -> 문장 완성: 오늘 서울에서 · 가 개최되었다.
3. 토큰: ##에 (점수: 0.0090) -> 문장 완성: 오늘 서울에서에 가 개최되었다.
4. 토큰: ##의 (점수: 0.0037) -> 문장 완성: 오늘 서울에서의 가 개최되었다.
5. 토큰: ##로 (점수: 0.0035) -> 문장 완성: 오늘 서울에서로 가 개최되었다.


In [62]:
# Tokenizer 학습
# 16. 마스크드 언어 모델(MLM) 추론 테스트 코드
# - 추론 코드 직접 작성, 토큰화/배치/최적화 제어를 위한 유연성 필요하기 때문

import torch
from transformers import BertTokenizer, BertForMaskedLM

# 1. 토크나이저/모델 직접 로드
model_path = "./content/bert_masked_lm_ckpt/checkpoint-282" # 가장 마지막 체크포인트 경로
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForMaskedLM.from_pretrained(model_path)

# 미분 계산이 필요 없으므로 반드시 평가 모드로 전환하고 그라디언트 비활성화
model.eval()

text = "오늘 서울에서 [MASK]가 개최되었다."

# 2. 토크나이징: PyTorch 텐서 형태로 변환
inputs = tokenizer(text=text, return_tensors="pt")

# 3. 순전파(Forward) 실행: torch.no_grad()로 미분 게산(메모리) 차단
with torch.no_grad():
    # 모델에 입력값을 넣어 결과값을 얻는다
    outputs = model(**inputs)
    # outputs.logits의 형태: (batch_size, sequence_length, vocab_size)
    logits = outputs.logits

# 4. [MASK] 토큰의 위치(Index) 찾기
mask_token_id = tokenizer.mask_token_id
print(f"mask_token_id: {mask_token_id}")

# input_ids에서 마스크 토큰이 어디에 있는지 위치(True/False)를 찾고 인덱스 추출
mask_token_index = torch.where(inputs["input_ids"] == mask_token_id)[1]
print(f"mask_token_index: {mask_token_index}")

# 5. 해당 [MASK] 위치의 로직스(logits, 3만여개 단어에 대한 점수) 추출
mask_logits = logits[0, mask_token_index, :]
print(f"mask_logits: {mask_logits}")

# 6. 상위 k개의 후보 단어 추출: Top-5
topk = 5
topk_values, topk_indices = torch.topk(mask_logits, topk, dim=-1)

# 7. 디코딩 및 결과 확인
print(f"입력 문장: {text}\n")
for i in range(topk):
    token_id = topk_indices[0][i].item()
    score = topk_values[0][i].item()
    token_str = tokenizer.decode([token_id])

    # [MASK] 토큰을 예측된 단어로 치환하여 완성된 문장 만들기
    # input_ids를 리스트로 복사해서 값 교체 후 디코딩
    filled_ids = inputs["input_ids"][0].clone()
    filled_ids[mask_token_index] = token_id
    completed_sentence = tokenizer.decode(filled_ids, skip_special_tokens=True)

    print(f"{i+1}. 토큰: {token_str} (점수: {score:.4f}) -> 완성 문장: {completed_sentence}")

loading file vocab.txt
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading file tokenizer.json
loading configuration file ./content/bert_masked_lm_ckpt/checkpoint-282/config.json
Model config BertConfig {
  "_name_or_path": "./content/bert_masked_lm_ckpt/checkpoint-189",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.44.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 24000
}

loading weights file ./content/bert_masked_lm_ckpt/checkpoint-282/model.safetenso

mask_token_id: 4
mask_token_index: tensor([4])
mask_logits: tensor([[-4.2772, -3.8694, -2.3741,  ..., -3.8493, -4.8077, -4.2018]])
입력 문장: 오늘 서울에서 [MASK]가 개최되었다.

1. 토큰: … (점수: 3.4916) -> 완성 문장: 오늘 서울에서 … 가 개최되었다.
2. 토큰: · (점수: 2.5570) -> 완성 문장: 오늘 서울에서 · 가 개최되었다.
3. 토큰: ##에 (점수: 1.7511) -> 완성 문장: 오늘 서울에서에 가 개최되었다.
4. 토큰: 美 (점수: 1.7158) -> 완성 문장: 오늘 서울에서 美 가 개최되었다.
5. 토큰: ##서 (점수: 1.4626) -> 완성 문장: 오늘 서울에서서 가 개최되었다.


In [ ]:
# Evaluate 라이브러리
# 1. Evaluate 평가
import evaluate # 평가 지표 라이브러리

# Evaluate 클래스 평가 지표 로드
acc = evaluate.load("accuracy")

# combine 함수: 정확도/f1스코어/정밀도/재현율 동시에 가져오면 딕셔너리 형태로 반환
metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

# 예1) 더미데이터 파라미터인 predictions는 모델 추론 결과를 입력하고 references는 실제 결과값을 입력
# {'accuracy': 0.5, 'f1': 0.5, 'precision': 0.5, 'recall': 0.5}
# metrics.compute(predictions=[1,0,0,1], references=[0,1,0,1])

# 예2) add 메서드 사용: 값을 저장한 후에 compute 한번에 계산, 데이터를 하나씩 샘플 단위/ 배치 단위 여러개씩 저장 할 수 있음
# {'accuracy': 0.5, 'f1': 0.5, 'precision': 0.5, 'recall': 0.5}
# 데이터 스트리링이나 대용량 검증셋(Validation Set)을 평가
for y, pred in zip([0,1,0,1], [1,0,0,1]):
    metrics.add(prediction=pred, reference=y)
metrics.compute()

# 예3) add_batch 매서드를 사용: 배치 단위로 데이터를 올린다, 
# 실무에서는 배치 단위(리스트,텐서,배열), DataLoader 등으로 묶인 배치 단위 데이터에 유용하다
# {'accuracy': 0.5, 'f1': 0.5, 'precision': 0.5, 'recall': 0.5}
for y, pred in zip([[0,1],[0,1]], [[1,0],[0,1]]):
    metrics.add_batch(predictions=pred, references=y)
metrics.compute()

{'accuracy': 0.5, 'f1': 0.5, 'precision': 0.5, 'recall': 0.5}

In [16]:
# Evaluate 라이브러리
# 2. 커스텀 메트릭
import evaluate

# combine 함수: 정확도/f1스코어/정밀도/재현율 동시에 가져오면 딕셔너리 형태로 반환
metrics = evaluate.combine(["accuracy","f1","precision","recall"])

def simple_accuracy(preds, labels):
    # 순서1. (preds == labels) 요소별 비교: 값이 같으면 True 다르면 False (예: [True, False, False, True])
    # 순서2. .to(float) 숫자형 변환: True 1.0, False 0.0 자동 변환 (예: [1.0, 0.0, 0.0, 1.0])
    # 순서3. .mean() 평균 계산 = 정확도: 변환된 1과 0들의 평균값
    # 순서4. .item() 파이썬 기본 숫자로 추출: 파이토치 텐서 형태로 남아 있는 값(tensor(0.5)) -> float 0.5 로 꺼낸다
    # 순서5. return {"accuracy": 딕셔너리 형태로 반환
    return {"accuracy": (preds == labels).to(float).mean().item()}

In [ ]:
# Evaluate 라이브러리
# 3. Trainer 적용 